In [1]:
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor

In [2]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model= NeuralNetwork()
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size= len(dataloader.dataset)
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        X, y= X.to(device), y.to(device)

        pred= model(X)
        loss= loss_fn(pred, y)

        
        loss.backward()
        optimizer.step()    
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current= loss.item(), batch*len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [ ]:
def test_loop(dataloader, model, loss_fn):
    size= len(dataloader.dataset)
    num_batches= len(dataloader)
    model.eval() # parametrelerin güncellenmemesi için
    test_loss, correct= 0,0

    with torch.no_grad():
        for X,y in dataloader:
            X, y= X.to(device), y.to(device)
            pred= model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")